# APWC strategy backtest suite

このノートブックは、wide 型で与えられた以下の3データから、テーマバスケット戦略を比較バックテストするための実装です。

- `apwc`: テーマバスケットの APWC 時系列
- `total_ret`: テーマバスケットのトータルリターン系列
- `resid_ret`: テーマバスケットの残差リターン系列

全ての戦略は、以下を基本制約とします。

- ロングオンリー
- 候補がある場合はテーマバスケットに100%投資
- 候補がない場合は100%キャッシュ
- キャッシュリターンは0%
- モメンタム系戦略は `signal > 0` のテーマだけを候補にする
- モメンタム系戦略では、選ばれたテーマを等ウェイトにせず、正のリスク調整済みモメンタムスコアに比例してウェイトを付ける

実装済み戦略は以下の11本です。

| No. | Strategy |
|---:|---|
| 1 | Equal Weight |
| 2 | Total Return Momentum |
| 3 | Residual Return Momentum |
| 4 | RRG High APWC Total Momentum |
| 5 | RRG High APWC Residual Momentum |
| 6 | RRG High APWC Total Momentum + RRG Decay Exclusion |
| 7 | RRG High APWC Residual Momentum + RRG Decay Exclusion |
| 8 | Threshold High APWC Total Momentum |
| 9 | Threshold High APWC Residual Momentum |
| 10 | Threshold High APWC Total Momentum + RRG Decay Exclusion |
| 11 | Threshold High APWC Residual Momentum + RRG Decay Exclusion |

## 0. Definitions

APWC rolling z-score:

\[
Z_{t,i} = \frac{APWC_{t,i} - \mu^{(L)}_{t,i}}{\sigma^{(L)}_{t,i}}
\]

RRG coherence coordinates:

\[
X_{t,i} = EMA_{20}(Z_{t,i})
\]

\[
Y_{t,i} = EMA_{10}(Z_{t,i}) - EMA_{40}(Z_{t,i})
\]

RRG High APWC gate:

\[
G^{RRG}_{t,i} = 1[X_{t,i} > 0] \cdot 1[APWC_{t,i} > 0]
\]

Threshold High APWC gate:

\[
G^{THR}_{t,i}(\theta)=1[Z_{t,i}\geq\theta]\cdot1[APWC_{t,i}>0]
\]

RRG decay exclusion:

\[
D^{decay}_{t,i}
=
1[X_{t,i}>0]\cdot1[Y_{t,i}<0]\cdot1[\Delta X_{t,i}<0]\cdot1[E_{t,i}=1]
\]

where \(E_{t,i}\) means the theme experienced an `Expanding` state \((X>0, Y>0)\) during the recent lookback window.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.options.display.float_format = "{:.6f}".format

## 1. User settings

実データで実行する場合は `USE_SAMPLE_DATA = False` にして、3つのファイルパスを指定してください。CSV / Excel / Parquet に対応しています。

In [ ]:
# -----------------------------
# Data settings
# -----------------------------
USE_SAMPLE_DATA = True

APWC_PATH = Path("apwc.csv")
TOTAL_RET_PATH = Path("total_ret.csv")
RESID_RET_PATH = Path("resid_ret.csv")

APWC_SHEET = 0
TOTAL_RET_SHEET = 0
RESID_RET_SHEET = 0
DATE_COL = 0

# -----------------------------
# Backtest settings
# -----------------------------
REBALANCE_FREQ = "W-FRI"  # None or "D" for daily, "W-FRI" for weekly, "M" for monthly
CASH_RETURN = 0.0
TRANSACTION_COST_BPS = 0.0

# Momentum settings
MOM_WINDOW = 60
VOL_WINDOW = 60
SCORE_CLIP_LOWER = -3.0
SCORE_CLIP_UPPER = 3.0
TOP_N = 30  # None means use all positive-signal candidates

# APWC / RRG settings
APWC_Z_WINDOW = 252
RRG_LEVEL_SPAN = 20
RRG_MOM_SHORT_SPAN = 10
RRG_MOM_LONG_SPAN = 40
THRESHOLD_Z = 1.5

# Decay settings
DECAY_LOOKBACK = 60
DECAY_DELTA_HORIZON = 5

# Output settings
OUTPUT_DIR = Path("outputs_strategy_suite")
SAVE_OUTPUTS = False

# Optional robustness. Keep False for fast default execution.
RUN_THRESHOLD_ROBUSTNESS = False
THRESHOLD_GRID = [1.0, 1.5, 2.0]

## 2. Data utilities

In [ ]:
def read_wide(path: str | Path,
              date_col: int | str = 0,
              sheet_name: int | str = 0) -> pd.DataFrame:
    """Read a wide DataFrame with dates as index and themes as columns."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path, index_col=date_col, parse_dates=True)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, index_col=date_col, parse_dates=True, sheet_name=sheet_name)
    elif suffix == ".parquet":
        df = pd.read_parquet(path)
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df.loc[~df.index.duplicated(keep="last")]
    df = df.apply(pd.to_numeric, errors="coerce")
    return df


def validate_wide(df: pd.DataFrame, name: str) -> None:
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{name} must be a pandas DataFrame.")
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}.index must be a DatetimeIndex.")
    if df.index.duplicated().any():
        raise ValueError(f"{name} contains duplicated dates.")
    if df.columns.duplicated().any():
        dupes = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"{name} contains duplicated columns: {dupes[:5]}")


def align_wide(apwc: pd.DataFrame,
               total_ret: pd.DataFrame,
               resid_ret: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Align dates and columns across the three wide DataFrames."""
    validate_wide(apwc, "apwc")
    validate_wide(total_ret, "total_ret")
    validate_wide(resid_ret, "resid_ret")

    idx = apwc.index.intersection(total_ret.index).intersection(resid_ret.index)
    cols = apwc.columns.intersection(total_ret.columns).intersection(resid_ret.columns)

    if len(idx) == 0:
        raise ValueError("No common dates across the input DataFrames.")
    if len(cols) == 0:
        raise ValueError("No common theme columns across the input DataFrames.")

    return (
        apwc.loc[idx, cols].sort_index(),
        total_ret.loc[idx, cols].sort_index(),
        resid_ret.loc[idx, cols].sort_index(),
    )

## 3. Core calculations

In [ ]:
def rolling_z(df: pd.DataFrame,
              window: int,
              min_periods: Optional[int] = None,
              ddof: int = 0) -> pd.DataFrame:
    if min_periods is None:
        min_periods = window
    mu = df.rolling(window, min_periods=min_periods).mean()
    sig = df.rolling(window, min_periods=min_periods).std(ddof=ddof)
    return (df - mu) / sig.replace(0.0, np.nan)


def ema(df: pd.DataFrame,
        span: int,
        min_periods: Optional[int] = None) -> pd.DataFrame:
    if min_periods is None:
        min_periods = span
    return df.ewm(span=span, adjust=False, min_periods=min_periods).mean()


def risk_adjusted_momentum(ret: pd.DataFrame,
                           mom_window: int = 60,
                           vol_window: int = 60,
                           clip_lower: float = -3.0,
                           clip_upper: float = 3.0) -> pd.DataFrame:
    """Risk-adjusted momentum score: trailing sum / trailing vol / sqrt(window)."""
    mom = ret.rolling(mom_window, min_periods=mom_window).sum()
    vol = ret.rolling(vol_window, min_periods=vol_window).std(ddof=0)
    score = mom / (vol * np.sqrt(mom_window)).replace(0.0, np.nan)
    return score.clip(lower=clip_lower, upper=clip_upper)


def compute_rrg_features(apwc: pd.DataFrame,
                         z_window: int = 252,
                         level_span: int = 20,
                         mom_short_span: int = 10,
                         mom_long_span: int = 40,
                         decay_lookback: int = 60,
                         decay_delta_horizon: int = 5) -> Dict[str, pd.DataFrame]:
    """Compute APWC z-score, RRG level X, momentum Y, RRG gate, and RRG decay flag."""
    z = rolling_z(apwc, z_window)
    x = ema(z, level_span)
    y = ema(z, mom_short_span) - ema(z, mom_long_span)

    expanding = (x > 0.0) & (y > 0.0)

    # Expanding state experienced in [t - decay_lookback, t - 1]
    recent_expanding = expanding.shift(1).rolling(
        decay_lookback, min_periods=1
    ).max().fillna(0.0).astype(bool)

    delta_x = x - x.shift(decay_delta_horizon)

    rrg_gate = (x > 0.0) & (apwc > 0.0)

    decay = (
        (x > 0.0)
        & (y < 0.0)
        & (delta_x < 0.0)
        & recent_expanding
    )

    states = pd.DataFrame(index=apwc.index, columns=apwc.columns, dtype="object")
    states[(x <= 0.0) & (y > 0.0)] = "Forming"
    states[(x > 0.0) & (y > 0.0)] = "Expanding"
    states[(x > 0.0) & (y <= 0.0)] = "Weakening"
    states[(x <= 0.0) & (y <= 0.0)] = "Dormant"

    return {
        "apwc_z": z,
        "rrg_x": x,
        "rrg_y": y,
        "expanding": expanding,
        "recent_expanding": recent_expanding,
        "delta_x": delta_x,
        "rrg_gate": rrg_gate,
        "rrg_decay": decay,
        "rrg_state": states,
    }


def threshold_gate(apwc: pd.DataFrame,
                   apwc_z: pd.DataFrame,
                   threshold: float = 1.5) -> pd.DataFrame:
    return (apwc_z >= threshold) & (apwc > 0.0)

## 4. Weighting and backtest engine

モメンタム系戦略では、候補テーマの中で正の risk-adjusted momentum score に比例してウェイトを付けます。

\[
w_{t,i}=\frac{\max(S_{t,i},0)}{\sum_{j\in C_t}\max(S_{t,j},0)}
\]

候補がゼロの場合、全テーマウェイトは0となり、100%キャッシュ扱いになります。

In [ ]:
def keep_top_n(score: pd.DataFrame, top_n: Optional[int] = 30) -> pd.DataFrame:
    """Keep only top N positive scores per row. If top_n is None, keep all positive scores."""
    positive = score.where(score > 0.0, 0.0).fillna(0.0)
    if top_n is None:
        return positive

    ranks = positive.where(positive > 0.0).rank(axis=1, method="first", ascending=False)
    kept = positive.where((positive > 0.0) & (ranks <= top_n), 0.0)
    return kept.fillna(0.0)


def normalize_scores_to_weights(score: pd.DataFrame,
                                top_n: Optional[int] = 30) -> pd.DataFrame:
    """Convert positive scores to long-only weights that sum to 1 when candidates exist."""
    kept = keep_top_n(score, top_n=top_n)
    denom = kept.sum(axis=1).replace(0.0, np.nan)
    weights = kept.div(denom, axis=0).fillna(0.0)
    return weights


def equal_weight_from_available(total_ret: pd.DataFrame) -> pd.DataFrame:
    available = total_ret.notna()
    count = available.sum(axis=1).replace(0, np.nan)
    return available.astype(float).div(count, axis=0).fillna(0.0)


def apply_rebalance(raw_weights: pd.DataFrame,
                    rebalance_freq: Optional[str] = "W-FRI") -> pd.DataFrame:
    """Hold raw weights until the next rebalance date."""
    if rebalance_freq is None or str(rebalance_freq).upper() == "D":
        return raw_weights.fillna(0.0)

    # Last available trading date in each resample period.
    idx_series = pd.Series(raw_weights.index, index=raw_weights.index)
    rebal_dates = pd.DatetimeIndex(idx_series.resample(rebalance_freq).last().dropna().values)
    rebal_dates = raw_weights.index.intersection(rebal_dates)

    if len(rebal_dates) == 0:
        return raw_weights * 0.0

    held = raw_weights.loc[rebal_dates].reindex(raw_weights.index).ffill().fillna(0.0)
    return held


def weighted_average(weights: pd.DataFrame,
                     values: pd.DataFrame) -> pd.Series:
    """Row-wise weighted average. Returns NaN when total weight is zero."""
    w = weights.reindex_like(values).fillna(0.0)
    v = values.reindex_like(weights)
    denom = w.sum(axis=1).replace(0.0, np.nan)
    return (w * v).sum(axis=1) / denom


@dataclass
class BacktestResult:
    name: str
    weights: pd.DataFrame
    pnl: pd.DataFrame
    diagnostics: pd.DataFrame
    raw_weights: pd.DataFrame
    raw_candidate_mask: pd.DataFrame


def backtest_from_weights(name: str,
                          raw_weights: pd.DataFrame,
                          total_ret: pd.DataFrame,
                          resid_ret: pd.DataFrame,
                          apwc_z: Optional[pd.DataFrame] = None,
                          rrg_x: Optional[pd.DataFrame] = None,
                          rrg_y: Optional[pd.DataFrame] = None,
                          raw_candidate_mask: Optional[pd.DataFrame] = None,
                          raw_decay_excluded_mask: Optional[pd.DataFrame] = None,
                          rebalance_freq: Optional[str] = "W-FRI",
                          transaction_cost_bps: float = 0.0,
                          cash_return: float = 0.0) -> BacktestResult:
    """Apply rebalance, compute total/residual/common PnL, cost, cash diagnostics."""
    total_ret = total_ret.reindex_like(raw_weights)
    resid_ret = resid_ret.reindex_like(raw_weights)

    weights = apply_rebalance(raw_weights, rebalance_freq=rebalance_freq)
    lagged_weights = weights.shift(1).fillna(0.0)

    invested_weight_lagged = lagged_weights.sum(axis=1).clip(lower=0.0, upper=1.0)
    cash_weight_lagged = 1.0 - invested_weight_lagged

    # Missing returns are treated as 0 after alignment. Prefer complete input data in production.
    total_pnl_gross = (lagged_weights * total_ret.fillna(0.0)).sum(axis=1) + cash_weight_lagged * cash_return
    resid_pnl_gross = (lagged_weights * resid_ret.fillna(0.0)).sum(axis=1)
    common_pnl_gross = total_pnl_gross - resid_pnl_gross

    turnover_signal_date = weights.diff().abs().sum(axis=1).fillna(weights.abs().sum(axis=1))
    # Weights decided at t are implemented for t+1. Deduct transaction cost on t+1.
    exec_turnover = turnover_signal_date.shift(1).fillna(0.0)
    cost = exec_turnover * (transaction_cost_bps / 10_000.0)

    pnl = pd.DataFrame({
        "total_gross": total_pnl_gross,
        "residual_gross": resid_pnl_gross,
        "common_gross": common_pnl_gross,
        "turnover": exec_turnover,
        "cost": cost,
        "total_net": total_pnl_gross - cost,
        "residual_net": resid_pnl_gross - cost,
        "common_net": common_pnl_gross,
    })

    invested_weight = weights.sum(axis=1).clip(lower=0.0, upper=1.0)
    cash_weight = 1.0 - invested_weight
    selected_count = (weights > 0.0).sum(axis=1)
    max_weight = weights.max(axis=1)
    herfindahl = (weights ** 2).sum(axis=1)

    if raw_candidate_mask is None:
        raw_candidate_mask = raw_weights > 0.0
    candidate_count = raw_candidate_mask.sum(axis=1).reindex(weights.index)

    diagnostics = pd.DataFrame({
        "invested_weight": invested_weight,
        "cash_weight": cash_weight,
        "selected_count": selected_count,
        "candidate_count": candidate_count,
        "max_weight": max_weight,
        "herfindahl": herfindahl,
        "turnover": exec_turnover,
    })

    if raw_decay_excluded_mask is not None:
        diagnostics["decay_excluded_count"] = raw_decay_excluded_mask.sum(axis=1).reindex(weights.index)
    else:
        diagnostics["decay_excluded_count"] = 0.0

    if apwc_z is not None:
        diagnostics["avg_held_apwc_z"] = weighted_average(weights, apwc_z)
    if rrg_x is not None:
        diagnostics["avg_held_rrg_x"] = weighted_average(weights, rrg_x)
    if rrg_y is not None:
        diagnostics["avg_held_rrg_y"] = weighted_average(weights, rrg_y)

    return BacktestResult(
        name=name,
        weights=weights,
        pnl=pnl,
        diagnostics=diagnostics,
        raw_weights=raw_weights,
        raw_candidate_mask=raw_candidate_mask,
    )


def build_momentum_raw_weights(signal_score: pd.DataFrame,
                               gate: Optional[pd.DataFrame] = None,
                               exclusion: Optional[pd.DataFrame] = None,
                               top_n: Optional[int] = 30) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Create raw daily weights from positive signal, optional gate, and optional exclusion."""
    score = signal_score.copy()
    candidate_mask = score > 0.0

    if gate is not None:
        candidate_mask = candidate_mask & gate.reindex_like(score).fillna(False)

    excluded_mask = pd.DataFrame(False, index=score.index, columns=score.columns)
    if exclusion is not None:
        excluded_mask = exclusion.reindex_like(score).fillna(False)
        candidate_mask = candidate_mask & (~excluded_mask)

    allocation_score = score.where(candidate_mask, 0.0).clip(lower=0.0)
    raw_weights = normalize_scores_to_weights(allocation_score, top_n=top_n)

    # Candidate mask after strategy-specific conditions, before TOP_N. Useful for diagnostics.
    return raw_weights, candidate_mask, excluded_mask

## 5. Performance and diagnostics

In [ ]:
def drawdown_series(ret: pd.Series) -> pd.Series:
    ret = ret.fillna(0.0)
    wealth = (1.0 + ret).cumprod()
    return wealth / wealth.cummax() - 1.0


def performance_stats(ret: pd.Series, ann_factor: int = 252) -> pd.Series:
    ret = ret.fillna(0.0)
    if len(ret) == 0:
        return pd.Series(dtype=float)

    wealth = (1.0 + ret).cumprod()
    dd = wealth / wealth.cummax() - 1.0
    avg = ret.mean()
    vol = ret.std(ddof=0)
    ann_vol = vol * np.sqrt(ann_factor)
    ann_return = wealth.iloc[-1] ** (ann_factor / len(ret)) - 1.0
    downside = ret.where(ret < 0.0, 0.0).std(ddof=0) * np.sqrt(ann_factor)

    return pd.Series({
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": np.nan if vol == 0.0 else avg / vol * np.sqrt(ann_factor),
        "sortino": np.nan if downside == 0.0 else ann_return / downside,
        "max_drawdown": dd.min(),
        "calmar": np.nan if dd.min() == 0.0 else ann_return / abs(dd.min()),
        "daily_hit_rate": (ret > 0.0).mean(),
        "avg_daily_return": avg,
        "daily_vol": vol,
        "skew": ret.skew(),
        "kurtosis": ret.kurtosis(),
        "cumulative_return": wealth.iloc[-1] - 1.0,
        "n_days": len(ret),
    })


def diagnostic_stats(result: BacktestResult) -> pd.Series:
    d = result.diagnostics.copy()
    return pd.Series({
        "cash_ratio": (d["cash_weight"] >= 0.999999).mean(),
        "avg_invested_weight": d["invested_weight"].mean(),
        "avg_cash_weight": d["cash_weight"].mean(),
        "candidate_count_mean": d["candidate_count"].mean(),
        "candidate_count_median": d["candidate_count"].median(),
        "zero_candidate_days": (d["candidate_count"] == 0).sum(),
        "selected_count_mean": d["selected_count"].mean(),
        "selected_count_median": d["selected_count"].median(),
        "avg_max_weight": d["max_weight"].mean(),
        "avg_herfindahl": d["herfindahl"].mean(),
        "effective_n_mean": np.nan if d["herfindahl"].replace(0, np.nan).empty else (1.0 / d["herfindahl"].replace(0, np.nan)).mean(),
        "avg_daily_turnover": d["turnover"].mean(),
        "decay_excluded_count_mean": d["decay_excluded_count"].mean(),
        "avg_held_apwc_z": d["avg_held_apwc_z"].mean() if "avg_held_apwc_z" in d else np.nan,
        "avg_held_rrg_x": d["avg_held_rrg_x"].mean() if "avg_held_rrg_x" in d else np.nan,
        "avg_held_rrg_y": d["avg_held_rrg_y"].mean() if "avg_held_rrg_y" in d else np.nan,
    })


def summarize_results(results: Dict[str, BacktestResult],
                      return_col: str = "total_net") -> pd.DataFrame:
    rows = []
    for name, result in results.items():
        perf = performance_stats(result.pnl[return_col])
        diag = diagnostic_stats(result)
        row = pd.concat([perf, diag])
        row.name = name
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_total_and_residual(results: Dict[str, BacktestResult]) -> pd.DataFrame:
    rows = []
    for name, result in results.items():
        total = performance_stats(result.pnl["total_net"]).add_prefix("total_")
        resid = performance_stats(result.pnl["residual_net"]).add_prefix("resid_")
        diag = diagnostic_stats(result)
        row = pd.concat([total, resid, diag])
        row.name = name
        rows.append(row)
    return pd.DataFrame(rows)

## 6. Plot utilities

In [ ]:
def plot_cumulative_returns(results: Dict[str, BacktestResult],
                            names: Optional[Iterable[str]] = None,
                            return_col: str = "total_net",
                            title: str = "Cumulative returns") -> None:
    if names is None:
        names = results.keys()
    fig, ax = plt.subplots(figsize=(12, 6))
    for name in names:
        ret = results[name].pnl[return_col].fillna(0.0)
        ((1.0 + ret).cumprod() - 1.0).plot(ax=ax, label=name)
    ax.set_title(title)
    ax.set_ylabel("Cumulative return")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_drawdowns(results: Dict[str, BacktestResult],
                   names: Optional[Iterable[str]] = None,
                   return_col: str = "total_net",
                   title: str = "Drawdowns") -> None:
    if names is None:
        names = results.keys()
    fig, ax = plt.subplots(figsize=(12, 6))
    for name in names:
        drawdown_series(results[name].pnl[return_col]).plot(ax=ax, label=name)
    ax.set_title(title)
    ax.set_ylabel("Drawdown")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_cash_ratio(results: Dict[str, BacktestResult],
                    names: Optional[Iterable[str]] = None,
                    title: str = "Cash weight") -> None:
    if names is None:
        names = results.keys()
    fig, ax = plt.subplots(figsize=(12, 4))
    for name in names:
        results[name].diagnostics["cash_weight"].plot(ax=ax, label=name)
    ax.set_title(title)
    ax.set_ylabel("Cash weight")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.show()

## 7. Sample data generator

ノートブックの動作確認用です。実証結果を示すものではありません。

In [ ]:
def make_sample_data(n_days: int = 1_250,
                     n_themes: int = 45,
                     seed: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Synthetic sample data for end-to-end notebook execution."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2019-01-01", periods=n_days)
    themes = [f"Theme_{i:02d}" for i in range(1, n_themes + 1)]

    market = rng.normal(0.0002, 0.006, size=(n_days, 1))
    common = rng.normal(0.0, 0.004, size=(n_days, n_themes))
    resid = rng.normal(0.0, 0.008, size=(n_days, n_themes))
    total = 0.35 * market + common + resid
    apwc = rng.normal(0.0, 0.008, size=(n_days, n_themes))

    for j in range(n_themes):
        n_episodes = rng.integers(1, 4)
        for _ in range(n_episodes):
            start = int(rng.integers(180, max(181, n_days - 220)))
            length = int(rng.integers(60, 170))
            end = min(n_days, start + length)
            direction = rng.choice([-1.0, 1.0])

            # Coherence build and decay profile
            profile = np.sin(np.linspace(0, np.pi, end - start))
            apwc[start:end, j] += 0.06 * profile + rng.normal(0.0, 0.006, size=end - start)

            # Residual momentum stronger during coherent periods
            drift = direction * rng.uniform(0.0005, 0.0012)
            resid[start:end, j] += drift
            for t in range(start + 1, end):
                resid[t, j] += 0.12 * resid[t - 1, j]

            # Total return can weaken late in very mature phase due to common drag
            late_start = start + int(0.70 * (end - start))
            total[start:end, j] += drift
            total[late_start:end, j] -= direction * rng.uniform(0.0003, 0.0009)

    apwc = pd.DataFrame(apwc, index=dates, columns=themes)
    total_ret = pd.DataFrame(total, index=dates, columns=themes)
    resid_ret = pd.DataFrame(resid, index=dates, columns=themes)
    return apwc, total_ret, resid_ret

## 8. Load data

In [ ]:
if USE_SAMPLE_DATA:
    apwc, total_ret, resid_ret = make_sample_data()
else:
    apwc = read_wide(APWC_PATH, date_col=DATE_COL, sheet_name=APWC_SHEET)
    total_ret = read_wide(TOTAL_RET_PATH, date_col=DATE_COL, sheet_name=TOTAL_RET_SHEET)
    resid_ret = read_wide(RESID_RET_PATH, date_col=DATE_COL, sheet_name=RESID_RET_SHEET)

apwc, total_ret, resid_ret = align_wide(apwc, total_ret, resid_ret)

print("apwc shape      :", apwc.shape)
print("total_ret shape :", total_ret.shape)
print("resid_ret shape :", resid_ret.shape)
print("date range      :", apwc.index.min().date(), "to", apwc.index.max().date())
print("n themes        :", len(apwc.columns))

display(apwc.head())

## 9. Compute signals, gates, and decay flags

In [ ]:
# Momentum scores
total_mom_score = risk_adjusted_momentum(
    total_ret,
    mom_window=MOM_WINDOW,
    vol_window=VOL_WINDOW,
    clip_lower=SCORE_CLIP_LOWER,
    clip_upper=SCORE_CLIP_UPPER,
)

resid_mom_score = risk_adjusted_momentum(
    resid_ret,
    mom_window=MOM_WINDOW,
    vol_window=VOL_WINDOW,
    clip_lower=SCORE_CLIP_LOWER,
    clip_upper=SCORE_CLIP_UPPER,
)

# RRG features
rrg = compute_rrg_features(
    apwc=apwc,
    z_window=APWC_Z_WINDOW,
    level_span=RRG_LEVEL_SPAN,
    mom_short_span=RRG_MOM_SHORT_SPAN,
    mom_long_span=RRG_MOM_LONG_SPAN,
    decay_lookback=DECAY_LOOKBACK,
    decay_delta_horizon=DECAY_DELTA_HORIZON,
)

apwc_z = rrg["apwc_z"]
rrg_x = rrg["rrg_x"]
rrg_y = rrg["rrg_y"]
rrg_gate = rrg["rrg_gate"]
rrg_decay = rrg["rrg_decay"]

thr_gate = threshold_gate(apwc, apwc_z, threshold=THRESHOLD_Z)

state_counts = rrg["rrg_state"].stack().value_counts(dropna=False)
print("RRG state counts")
display(state_counts)

print("Average RRG gate count:", rrg_gate.sum(axis=1).mean())
print("Average threshold gate count:", thr_gate.sum(axis=1).mean())
print("Average RRG decay count:", rrg_decay.sum(axis=1).mean())

## 10. Build the 11 strategies

In [ ]:
def run_equal_weight_strategy() -> BacktestResult:
    raw_w = equal_weight_from_available(total_ret)
    candidate_mask = total_ret.notna()
    return backtest_from_weights(
        name="01_Equal_Weight",
        raw_weights=raw_w,
        total_ret=total_ret,
        resid_ret=resid_ret,
        apwc_z=apwc_z,
        rrg_x=rrg_x,
        rrg_y=rrg_y,
        raw_candidate_mask=candidate_mask,
        rebalance_freq=REBALANCE_FREQ,
        transaction_cost_bps=TRANSACTION_COST_BPS,
        cash_return=CASH_RETURN,
    )


def run_momentum_strategy(name: str,
                          signal_score: pd.DataFrame,
                          gate: Optional[pd.DataFrame] = None,
                          exclusion: Optional[pd.DataFrame] = None) -> BacktestResult:
    raw_w, candidate_mask, excluded_mask = build_momentum_raw_weights(
        signal_score=signal_score,
        gate=gate,
        exclusion=exclusion,
        top_n=TOP_N,
    )
    return backtest_from_weights(
        name=name,
        raw_weights=raw_w,
        total_ret=total_ret,
        resid_ret=resid_ret,
        apwc_z=apwc_z,
        rrg_x=rrg_x,
        rrg_y=rrg_y,
        raw_candidate_mask=candidate_mask,
        raw_decay_excluded_mask=(excluded_mask & (signal_score > 0.0) & (gate if gate is not None else True)),
        rebalance_freq=REBALANCE_FREQ,
        transaction_cost_bps=TRANSACTION_COST_BPS,
        cash_return=CASH_RETURN,
    )


results: Dict[str, BacktestResult] = {}

results["01_Equal_Weight"] = run_equal_weight_strategy()

results["02_Total_Return_Momentum"] = run_momentum_strategy(
    "02_Total_Return_Momentum",
    signal_score=total_mom_score,
)

results["03_Residual_Return_Momentum"] = run_momentum_strategy(
    "03_Residual_Return_Momentum",
    signal_score=resid_mom_score,
)

results["04_RRG_High_APWC_Total_Momentum"] = run_momentum_strategy(
    "04_RRG_High_APWC_Total_Momentum",
    signal_score=total_mom_score,
    gate=rrg_gate,
)

results["05_RRG_High_APWC_Residual_Momentum"] = run_momentum_strategy(
    "05_RRG_High_APWC_Residual_Momentum",
    signal_score=resid_mom_score,
    gate=rrg_gate,
)

results["06_RRG_High_APWC_Total_Momentum_Decay_Exclusion"] = run_momentum_strategy(
    "06_RRG_High_APWC_Total_Momentum_Decay_Exclusion",
    signal_score=total_mom_score,
    gate=rrg_gate,
    exclusion=rrg_decay,
)

results["07_RRG_High_APWC_Residual_Momentum_Decay_Exclusion"] = run_momentum_strategy(
    "07_RRG_High_APWC_Residual_Momentum_Decay_Exclusion",
    signal_score=resid_mom_score,
    gate=rrg_gate,
    exclusion=rrg_decay,
)

results["08_Threshold_High_APWC_Total_Momentum"] = run_momentum_strategy(
    "08_Threshold_High_APWC_Total_Momentum",
    signal_score=total_mom_score,
    gate=thr_gate,
)

results["09_Threshold_High_APWC_Residual_Momentum"] = run_momentum_strategy(
    "09_Threshold_High_APWC_Residual_Momentum",
    signal_score=resid_mom_score,
    gate=thr_gate,
)

results["10_Threshold_High_APWC_Total_Momentum_Decay_Exclusion"] = run_momentum_strategy(
    "10_Threshold_High_APWC_Total_Momentum_Decay_Exclusion",
    signal_score=total_mom_score,
    gate=thr_gate,
    exclusion=rrg_decay,
)

results["11_Threshold_High_APWC_Residual_Momentum_Decay_Exclusion"] = run_momentum_strategy(
    "11_Threshold_High_APWC_Residual_Momentum_Decay_Exclusion",
    signal_score=resid_mom_score,
    gate=thr_gate,
    exclusion=rrg_decay,
)

print(f"Built {len(results)} strategies.")

## 11. Performance summary

主評価は `total_net` です。残差モメンタム系の診断として `residual_net` も併記しています。

In [ ]:
summary = summarize_total_and_residual(results)

key_cols = [
    "total_ann_return", "total_ann_vol", "total_sharpe", "total_max_drawdown", "total_cumulative_return",
    "resid_ann_return", "resid_ann_vol", "resid_sharpe", "resid_max_drawdown", "resid_cumulative_return",
    "cash_ratio", "candidate_count_mean", "selected_count_mean", "avg_daily_turnover",
    "decay_excluded_count_mean", "avg_held_apwc_z", "avg_held_rrg_x", "avg_held_rrg_y",
]

display(summary[key_cols].sort_values("total_sharpe", ascending=False))

## 12. Strategy comparison views

In [ ]:
all_names = list(results.keys())

plot_cumulative_returns(
    results,
    names=all_names,
    return_col="total_net",
    title="Total net cumulative returns: all strategies",
)

plot_drawdowns(
    results,
    names=all_names,
    return_col="total_net",
    title="Total net drawdowns: all strategies",
)

In [ ]:
# Focused comparisons
comparison_groups = {
    "Momentum vs Equal Weight": [
        "01_Equal_Weight",
        "02_Total_Return_Momentum",
        "03_Residual_Return_Momentum",
    ],
    "RRG gate effect": [
        "02_Total_Return_Momentum",
        "04_RRG_High_APWC_Total_Momentum",
        "03_Residual_Return_Momentum",
        "05_RRG_High_APWC_Residual_Momentum",
    ],
    "RRG decay effect": [
        "04_RRG_High_APWC_Total_Momentum",
        "06_RRG_High_APWC_Total_Momentum_Decay_Exclusion",
        "05_RRG_High_APWC_Residual_Momentum",
        "07_RRG_High_APWC_Residual_Momentum_Decay_Exclusion",
    ],
    "RRG vs threshold gates": [
        "04_RRG_High_APWC_Total_Momentum",
        "08_Threshold_High_APWC_Total_Momentum",
        "05_RRG_High_APWC_Residual_Momentum",
        "09_Threshold_High_APWC_Residual_Momentum",
        "07_RRG_High_APWC_Residual_Momentum_Decay_Exclusion",
        "11_Threshold_High_APWC_Residual_Momentum_Decay_Exclusion",
    ],
}

for title, names in comparison_groups.items():
    display(summary.loc[names, key_cols].sort_values("total_sharpe", ascending=False))
    plot_cumulative_returns(results, names=names, return_col="total_net", title=title)

## 13. Candidate count, cash ratio, and decay diagnostics

In [ ]:
diag_summary = pd.DataFrame({name: diagnostic_stats(result) for name, result in results.items()}).T

diag_cols = [
    "cash_ratio", "avg_invested_weight", "candidate_count_mean", "candidate_count_median",
    "zero_candidate_days", "selected_count_mean", "selected_count_median", "avg_max_weight",
    "avg_herfindahl", "effective_n_mean", "avg_daily_turnover", "decay_excluded_count_mean",
    "avg_held_apwc_z", "avg_held_rrg_x", "avg_held_rrg_y",
]

display(diag_summary[diag_cols])

plot_cash_ratio(
    results,
    names=[
        "04_RRG_High_APWC_Total_Momentum",
        "05_RRG_High_APWC_Residual_Momentum",
        "06_RRG_High_APWC_Total_Momentum_Decay_Exclusion",
        "07_RRG_High_APWC_Residual_Momentum_Decay_Exclusion",
        "08_Threshold_High_APWC_Total_Momentum",
        "09_Threshold_High_APWC_Residual_Momentum",
    ],
    title="Cash weight for APWC-gated strategies",
)

## 14. Pairwise comparison table

差分は `left - right` です。主に以下を確認します。

- residual momentum が total momentum に勝つか
- RRG gate が plain momentum を改善するか
- RRG decay exclusion が total return と drawdown を改善するか
- RRG gate が threshold gate より良いか

In [ ]:
def compare_pair(left: str, right: str) -> pd.Series:
    cols = [
        "total_ann_return", "total_sharpe", "total_max_drawdown", "total_cumulative_return",
        "resid_ann_return", "resid_sharpe", "resid_max_drawdown", "cash_ratio", "avg_daily_turnover",
    ]
    out = summary.loc[left, cols] - summary.loc[right, cols]
    out.name = f"{left} minus {right}"
    return out

pairs = [
    ("02_Total_Return_Momentum", "01_Equal_Weight"),
    ("03_Residual_Return_Momentum", "01_Equal_Weight"),
    ("03_Residual_Return_Momentum", "02_Total_Return_Momentum"),
    ("04_RRG_High_APWC_Total_Momentum", "02_Total_Return_Momentum"),
    ("05_RRG_High_APWC_Residual_Momentum", "03_Residual_Return_Momentum"),
    ("06_RRG_High_APWC_Total_Momentum_Decay_Exclusion", "04_RRG_High_APWC_Total_Momentum"),
    ("07_RRG_High_APWC_Residual_Momentum_Decay_Exclusion", "05_RRG_High_APWC_Residual_Momentum"),
    ("04_RRG_High_APWC_Total_Momentum", "08_Threshold_High_APWC_Total_Momentum"),
    ("05_RRG_High_APWC_Residual_Momentum", "09_Threshold_High_APWC_Residual_Momentum"),
    ("07_RRG_High_APWC_Residual_Momentum_Decay_Exclusion", "11_Threshold_High_APWC_Residual_Momentum_Decay_Exclusion"),
]

pairwise_comparison = pd.DataFrame([compare_pair(l, r) for l, r in pairs])
display(pairwise_comparison)

## 15. Threshold robustness

閾値型 High APWC gate は比較用です。必要な場合だけ `RUN_THRESHOLD_ROBUSTNESS = True` にしてください。

In [ ]:
def run_threshold_strategy_set(theta: float) -> Dict[str, BacktestResult]:
    gate = threshold_gate(apwc, apwc_z, threshold=theta)
    out = {}
    out[f"THR{theta}_Total"] = run_momentum_strategy(
        f"THR{theta}_Total", total_mom_score, gate=gate
    )
    out[f"THR{theta}_Residual"] = run_momentum_strategy(
        f"THR{theta}_Residual", resid_mom_score, gate=gate
    )
    out[f"THR{theta}_Total_Decay"] = run_momentum_strategy(
        f"THR{theta}_Total_Decay", total_mom_score, gate=gate, exclusion=rrg_decay
    )
    out[f"THR{theta}_Residual_Decay"] = run_momentum_strategy(
        f"THR{theta}_Residual_Decay", resid_mom_score, gate=gate, exclusion=rrg_decay
    )
    return out

if RUN_THRESHOLD_ROBUSTNESS:
    robust_results = {}
    for theta in THRESHOLD_GRID:
        robust_results.update(run_threshold_strategy_set(theta))
    threshold_robust_summary = summarize_total_and_residual(robust_results)
    display(threshold_robust_summary[key_cols].sort_values("total_sharpe", ascending=False))
else:
    robust_results = {}
    threshold_robust_summary = pd.DataFrame()
    print("RUN_THRESHOLD_ROBUSTNESS=False. Set it to True to run threshold robustness.")

## 16. Save outputs

`SAVE_OUTPUTS = True` にすると、主要な結果を CSV として保存します。

In [ ]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    summary.to_csv(OUTPUT_DIR / "strategy_summary_total_and_residual.csv")
    diag_summary.to_csv(OUTPUT_DIR / "strategy_diagnostics.csv")
    pairwise_comparison.to_csv(OUTPUT_DIR / "pairwise_comparison.csv")

    for name, result in results.items():
        safe_name = name.replace("/", "_").replace(" ", "_")
        result.weights.to_csv(OUTPUT_DIR / f"weights_{safe_name}.csv")
        result.pnl.to_csv(OUTPUT_DIR / f"pnl_{safe_name}.csv")
        result.diagnostics.to_csv(OUTPUT_DIR / f"diagnostics_{safe_name}.csv")

    if not threshold_robust_summary.empty:
        threshold_robust_summary.to_csv(OUTPUT_DIR / "threshold_robust_summary.csv")

    print(f"Saved outputs to {OUTPUT_DIR.resolve()}")
else:
    print("SAVE_OUTPUTS=False. Set it to True to save outputs.")

## 17. Interpretation guide

主な比較軸は以下です。

| 比較 | 解釈 |
|---|---|
| `02` vs `01` | Total return momentum が等ウェイトに勝つか |
| `03` vs `01` | Residual momentum が等ウェイトに勝つか |
| `03` vs `02` | residual signal が total signal より有効か |
| `04` vs `02` | RRG High APWC gate が total momentum を改善するか |
| `05` vs `03` | RRG High APWC gate が residual momentum を改善するか |
| `06` vs `04` | RRG decay exclusion が total momentum 戦略を改善するか |
| `07` vs `05` | RRG decay exclusion が residual momentum 戦略を改善するか |
| `04` vs `08` | RRG gate が threshold gate より良いか、total momentum で確認 |
| `05` vs `09` | RRG gate が threshold gate より良いか、residual momentum で確認 |
| `07` vs `11` | RRG gate + RRG decay が threshold gate + RRG decay より良いか |

今回の分析の主戦略候補は `07_RRG_High_APWC_Residual_Momentum_Decay_Exclusion` です。ただし、実データでの最終判断は、`total_net` の Sharpe、max drawdown、cash ratio、turnover、candidate count を同時に確認して行ってください。